# Cora 数据集加载 & 文本构造

逐个运行下面的单元格，逐步理解 `gen_data.py` 中 `get_data()` 的加载流程。

In [5]:
import os
import pandas as pd
import torch
import torch_geometric as pyg

print("导入成功！")

导入成功！


## Step 1: 加载 cora.pt 并查看基本信息

In [29]:
# 加载数据
cur_path = os.getcwd()  # Jupyter 中 __file__ 不可用，直接用当前工作目录
path = os.path.join(cur_path, "cora.pt")
data = torch.load(path)

print(f"数据类型: {type(data)}")
print(f"节点数: {data.num_nodes}")
print(f"原始边数: {data.num_edges}")
print(f"特征维度: {data.num_features}")
# num_classes 未存储在 cora.pt 中，从 y 标签推算
num_classes = data.y.max().item() + 1
print(f"类别数: {num_classes} (从 data.y 推算)")

数据类型: <class 'torch_geometric.data.data.Data'>
节点数: 2708
原始边数: 10858
特征维度: 384
类别数: 7 (从 data.y 推算)


In [30]:
# 查看原始文本和标签名
text = data.raw_texts
label_names = data.label_names
labels = data.y.tolist()  # 每个节点的标签索引

print(f"论文数量: {len(text)}  标签类别数: {len(label_names)}")
print(f"标签名: {label_names}")

# 用 pandas 展示前 5 条：索引、标题摘要（截断）、标签
df_text = pd.DataFrame({
    "节点ID": range(5),
    "标题+摘要 (前150字符)": [t[:150] + "..." for t in text[:5]],
    "标签": [label_names[labels[i]] for i in range(5)],
    "文本长度": [len(t) for t in text[:5]]
})
df_text

论文数量: 2708  标签类别数: 7
标签名: ['Rule_Learning', 'Neural_Networks', 'Case_Based', 'Genetic_Algorithms', 'Theory', 'Reinforcement_Learning', 'Probabilistic_Methods']


,节点ID,标题+摘要 (前150字符),标签,文本长度
0,0,Stochastic pro-positionalization of non-deter...,Rule_Learning,1008
1,1,Neural networks and statistical models. : ...,Neural_Networks,43
2,2,M.L. (1996) Design by Interactive Exploration...,Case_Based,1055
3,3,User\'s Guide to the PGAPack Parallel Genetic...,Genetic_Algorithms,80
4,4,Adaptive parameter pruning in neural networks...,Neural_Networks,972


In [31]:
# 查看边关系
print(f"边索引 shape: {data.edge_index.shape}")
print(f"节点数: {data.num_nodes}")
print(f"原始边数: {data.num_edges}")


def count_true_masks(masks):
    # print(f"mask 数量: {len(masks)}")

    total = 0

    for i, mask in enumerate(masks):
        true_count = mask.sum().item()
        # print(f"mask[{i}] True 数量: {true_count}")
        total += true_count

    # print(f"True 总数: {total}")
    return total


# 打印数据集划分
print(f"训练集: {len(data.train_masks), data.train_masks[0].shape}")
print(f"验证集: {len(data.val_masks), data.val_masks[0].shape}")
print(f"测试集: {len(data.test_masks), data.test_masks[0].shape}")
print(f"训练集节点数量: {count_true_masks(data.train_masks)}")
print(f"验证集节点数量: {count_true_masks(data.val_masks)}")
print(f"测试集节点数量: {count_true_masks(data.test_masks)}")


边索引 shape: torch.Size([2, 10858])
节点数: 2708
原始边数: 10858
训练集: (10, torch.Size([2708]))
验证集: (10, torch.Size([2708]))
测试集: (10, torch.Size([2708]))
训练集节点数量: 1400
验证集节点数量: 5000
测试集节点数量: 20680


`cora.pt` 保存的不是模型权重，而是一个完整的 `torch_geometric.data.Data` 图对象。实际包含：

| 字段 | 内容 |
|---|---|
| `x` | 节点特征，`float32 [2708, 384]` |
| `y` | 节点类别编号，`int64 [2708]`，取值 `0~6` |
| `edge_index` | 原始图边，`int64 [2, 10858]` |
| `raw_text` | 2708 篇论文的标题和摘要 |
| `raw_texts` | 与 `raw_text` 内容相同 |
| `label_names` | 7 个类别名称 |
| `category_names` | 每个节点对应的类别名称，共 2708 项 |
| `train_masks` | 10 组训练集掩码，每组 140 个节点 |
| `val_masks` | 10 组验证集掩码，每组 500 个节点 |
| `test_masks` | 10 组测试集掩码，每组 2068 个节点 |

在 [gen_data.py](D:/Bishe/GNN-Task_Relation/data/single_graph/Cora/gen_data.py:27) 中，加载后又做了两件事：

1. 取出 `raw_texts` 和 `label_names`。
2. 将原图转成无向 NetworkX 图，并在 [第 33 行](D:/Bishe/GNN-Task_Relation/data/single_graph/Cora/gen_data.py:33)复制所有字段、替换 `edge_index`。因此返回的 `new_data` 有 5278 条去重后的无向边，而 `cora.pt` 中仍是原始的 10858 个边条目。

后面生成的 `clean_text`、类别描述、各种 prompt 文本、task map 都不在 `cora.pt` 中；`num_nodes=2708`、`num_edges=10858` 和 `num_classes=7` 也是根据上述字段推导出来的属性。

## Step 2: 转换为 NetworkX 图并重建边索引（无向图）

In [ ]:
# 转为无向 NetworkX 图，再提取边索引
nx_g = pyg.utils.to_networkx(data, to_undirected=True)
edge_index = torch.tensor(list(nx_g.edges())).T

print(f"原始有向边数: {data.num_edges}")
print(f"无向图边数: {edge_index.size(1)}")
print(f"edge_index shape: {edge_index.size()}")
print()
print("前 5 条边 (src -> dst):")
print(edge_index[:, :5])

In [ ]:
# 用新的 edge_index 重建 Data 对象
data_dict = data.to_dict()
data_dict["edge_index"] = edge_index
new_data = pyg.data.data.Data(**data_dict)

print(f"new_data 节点数: {new_data.num_nodes}")
print(f"new_data 边数: {new_data.num_edges}")
print(f"new_data.y shape: {new_data.y.shape}")

## Step 3: 加载类别描述 CSV 并排序

In [ ]:
# 读取 categories.csv
category_desc = pd.read_csv(
    os.path.join(os.path.dirname("__file__") or os.getcwd(), "categories.csv"), sep=","
).values

print(f"category_desc shape: {category_desc.shape}")
print("列: [类别名, 类别描述]")
print()
for row in category_desc:
    print(f"  {row[0]}  ->  {row[1][:60]}...")

In [ ]:
# 按 label_names 顺序排列描述
ordered_desc = []
for i, label in enumerate(label_names):
    true_ind = label == category_desc[:, 0]
    ordered_desc.append((label, category_desc[:, 1][true_ind]))

print("有序类别描述:")
for desc in ordered_desc:
    print(f"  {desc[0]}  ->  {desc[1][0][:60]}...")

## Step 4: 构造各类文本

In [ ]:
# 4.1 特征节点文本 (论文标题+摘要)
clean_text = ["feature node. paper title and abstract: " + t for t in text]

print(f"clean_text 数量: {len(clean_text)}")
print("样例:")
print(clean_text[0][:200], "...")

In [ ]:
# 4.2 标签节点文本 (类别名+描述)
label_text = [
    "prompt node. literature category and description: "
    + desc[0]
    + "."
    + desc[1][0]
    for desc in ordered_desc
]

print(f"label_text 数量: {len(label_text)}")
for lt in label_text:
    print(f"  {lt[:100]}...")

In [ ]:
# 4.3 边标签文本 (链接预测: 有/无共引)
edge_label_text = [
    "prompt node. two papers do not have co-citation",
    "prompt node. two papers have co-citation"
]

print("edge_label_text:")
for et in edge_label_text:
    print(f"  {et}")

In [ ]:
# 4.4 逻辑标签文本 (OR / NOT AND 组合)
def get_logic_label(ordered_txt):
    or_labeled_text = []
    not_and_labeled_text = []
    for i in range(len(ordered_txt)):
        for j in range(len(ordered_txt)):
            c1 = ordered_txt[i]
            c2 = ordered_txt[j]
            txt = "prompt node. literature category and description: not " + c1[0] + ". " + c1[1][0] + " and not " + c2[0] + ". " + c2[1][0]
            not_and_labeled_text.append(txt)
            txt = "prompt node. literature category and description: either " + c1[0] + ". " + c1[1][0] + " or " + c2[0] + ". " + c2[1][0]
            or_labeled_text.append(txt)
    return or_labeled_text + not_and_labeled_text

logic_label_text = get_logic_label(ordered_desc)

print(f"logic_label_text 数量: {len(logic_label_text)}")
print("\n前 3 条 (OR):")
for t in logic_label_text[:3]:
    print(f"  {t[:120]}...")
print("\n第 N 条 (NOT AND):")
for t in logic_label_text[len(ordered_desc)**2: len(ordered_desc)**2 + 3]:
    print(f"  {t[:120]}...")

In [ ]:
# 4.5 边特征文本
edge_text = [
    "feature edge. connected papers are cited together by other papers."
]
print("edge_text:", edge_text)

In [ ]:
# 4.6 NOI 节点文本 (任务描述)
noi_node_edge_text = [
    "prompt node. link prediction on the papers that are cited together"
]
noi_node_text = [
    "prompt node. node classification on the paper's category"
]

print("noi_node_text [节点分类任务]:", noi_node_text)
print("noi_node_edge_text [链接预测任务]:", noi_node_edge_text)

In [ ]:
# 4.7 Prompt Edge 文本
prompt_edge_text = [
    "prompt edge",
    "prompt edge. edge for query graph that is our target",
    "prompt edge. edge for support graph that is an example"
]

print("prompt_edge_text:")
for i, pet in enumerate(prompt_edge_text):
    print(f"  [{i}] {pet}")

## Step 5: 构建任务描述字典 (Task Descriptions)

In [ ]:
task_dict = {
    # 端到端节点分类
    "e2e_node": {
        "noi_node_text_feat": ["noi_node_text_feat", [0]],
        "class_node_text_feat": ["class_node_text_feat", torch.arange(len(label_text))],
        "prompt_edge_text_feat": ["prompt_edge_text_feat", [0]]
    },
    # 端到端链接预测
    "e2e_link": {
        "noi_node_text_feat": ["noi_node_text_feat", [1]],
        "class_node_text_feat": ["class_node_text_feat",
                                   torch.arange(len(label_text), len(label_text) + len(edge_label_text))],
        "prompt_edge_text_feat": ["prompt_edge_text_feat", [0]]
    },
    # 少样本节点分类 (含 query/support edge)
    "lr_node": {
        "noi_node_text_feat": ["noi_node_text_feat", [0]],
        "class_node_text_feat": ["class_node_text_feat", torch.arange(len(label_text))],
        "prompt_edge_text_feat": ["prompt_edge_text_feat", [0, 1, 2]]
    },
    # 逻辑端到端 (OR / NOT AND 组合标签)
    "logic_e2e": {
        "noi_node_text_feat": ["noi_node_text_feat", [0]],
        "class_node_text_feat": ["class_node_text_feat",
                                   torch.arange(len(label_text) + len(edge_label_text),
                                                len(label_text) + len(edge_label_text) + len(logic_label_text))],
        "prompt_edge_text_feat": ["prompt_edge_text_feat", [0]]
    },
}

print("任务类型:")
for task_name, config in task_dict.items():
    print(f"\n  {task_name}:")
    for k, v in config.items():
        idx = v[1]
        if isinstance(idx, torch.Tensor):
            idx = idx.tolist()
        print(f"    {k}: 使用索引 {idx}")

## Step 6: 查看各文本列表的实际内容

In [ ]:
# 合并所有文本列表
all_text_feats = [
    clean_text,           # node_text_feat
    edge_text,            # edge_text_feat
    noi_node_text + noi_node_edge_text,  # noi_node_text_feat
    label_text + edge_label_text + logic_label_text,  # class_node_text_feat
    prompt_edge_text,     # prompt_edge_text_feat
]

names = ["node_text_feat", "edge_text_feat", "noi_node_text_feat", "class_node_text_feat", "prompt_edge_text_feat"]
for name, feat in zip(names, all_text_feats):
    print(f"{name}: 共 {len(feat)} 条")

print("\n--- 对应关系 ---")
print(f"node_text_feat[0]:         {clean_text[0][:80]}...")
print(f"edge_text_feat[0]:         {edge_text[0][:80]}...")
print(f"noi_node_text_feat[0]:     {noi_node_text[0][:80]}...")
print(f"noi_node_text_feat[1]:     {noi_node_edge_text[0][:80]}...")
print(f"class_node_text_feat[0]:   {label_text[0][:80]}...")
print(f"class_node_text_feat[{len(label_text)}]:  {edge_label_text[0][:80]}...")
print(f"class_node_text_feat[{len(label_text)+len(edge_label_text)}]: {logic_label_text[0][:80]}...")
print(f"prompt_edge_text_feat[0]:  {prompt_edge_text[0][:80]}...")

## Step 7: 调用原始的 get_data() 进行对比验证

In [ ]:
# 导入原始函数
from gen_data import get_data

graphs, text_feats, task_descs = get_data(None)  # dset 参数未被使用

print(f"图数量: {len(graphs)}")
print(f"图节点数: {graphs[0].num_nodes}")
print(f"图边数: {graphs[0].num_edges}")
print()
print(f"文本特征列表长度: {len(text_feats)}")
for i, tf in enumerate(text_feats):
    print(f"  [{i}] 长度: {len(tf)}")
print()
print(f"任务描述 keys: {list(task_descs.keys())}")